# PadelVision / TennisVision-AI — Pipeline Test
Run this on Google Colab (GPU runtime recommended) to test the full pipeline.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/Duggineniakhil/PadelVision.git
%cd PadelVision/backend

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics opencv-python-headless pandas scipy matplotlib gdown

In [ ]:
# 3. Download models from Google Drive
import gdown
import os

os.makedirs('models', exist_ok=True)

models = {
    'models/padel_ball_detector.pt': '15mzRe0BYYOzhyNH4sRldEnQFYSYGOVg6',
    'models/padel_court_keypoints.pt': '1xXZzpKD3BE5Fj636l-1NfBlRDhIX0uap',
}

for path, file_id in models.items():
    if not os.path.exists(path):
        print(f'Downloading {path}...')
        gdown.download(id=file_id, output=path, quiet=False)
    print(f'  OK: {path} ({os.path.getsize(path)/1e6:.1f} MB)')

In [ ]:
# 4. Upload a test video (or use the bundled one)
input_video = 'input_videos/input_video1.mp4'

if not os.path.exists(input_video):
    from google.colab import files
    print('No bundled video found. Upload a .mp4 file:')
    uploaded = files.upload()
    input_video = list(uploaded.keys())[0]
    print(f'Using uploaded video: {input_video}')
else:
    print(f'Using bundled video: {input_video}')

In [ ]:
# 5. Run the pipeline
import sys
import logging
sys.path.insert(0, '.')

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

from pipeline.processor import process_video

result = process_video(
    input_path=input_video,
    job_id='colab_test',
    on_progress=lambda stage, pct: print(f'  [{pct:>3}%] {stage}'),
)

print('\nDone!')
print(f'  Video   -> {result["video_path"]}')
print(f'  Stats   -> {result["analysis_path"]}')
print(f'  ShotMap -> {result["shot_map_path"]}')
print(f'  Highlights: {len(result["highlights"])} detected')

In [ ]:
# 6. Display analysis.json
import json

with open(result['analysis_path']) as f:
    analysis = json.load(f)

print(json.dumps(analysis, indent=2))

In [ ]:
# 7. Display heatmaps
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 4, figsize=(20, 10))
for i, ax in enumerate(axes, 1):
    path = result['heatmap_paths'].get(f'player_{i}')
    if path and os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(f'Player {i}')
    else:
        ax.text(0.5, 0.5, f'Player {i}\nNo data', ha='center', va='center')
    ax.axis('off')
plt.suptitle('Player Heatmaps', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Display shot map
shot_map_path = result['shot_map_path']
if os.path.exists(shot_map_path):
    img = mpimg.imread(shot_map_path)
    plt.figure(figsize=(6, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Ball Trajectory / Shot Map')
    plt.show()
else:
    print('Shot map not found')

In [ ]:
# 9. Show a sample frame from the annotated output video
import cv2
from IPython.display import display, Image as IPImage

cap = cv2.VideoCapture(result['video_path'])
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# Show frame at 30% of the video
cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * 0.3))
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(frame_rgb)
    plt.axis('off')
    plt.title(f'Annotated frame (frame {int(total * 0.3)} / {total})')
    plt.show()
else:
    print('Could not read output video')

In [ ]:
# 10. Download all outputs
from google.colab import files
import shutil

output_dir = f'outputs/colab_test'
shutil.make_archive('colab_test_output', 'zip', output_dir)
files.download('colab_test_output.zip')
print('Download started!')